# Prepare GtR data for Mission Radar

In [1]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import gtr
from discovery_utils.utils.io import safe_yaml_load

from discovery_utils.utils.llm import batch_check

from discovery_utils.utils import keywords as kw

PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [2]:
from datetime import datetime

def convert_to_date(x: str) -> str:
    try:
        return datetime.fromtimestamp(x / 1000).strftime('%Y-%m-%d')
    except:
        return ""

In [11]:
GTR = gtr.GtrGetter(data_version = "GtR_20250309")
# GTR = gtr.GtrGetter()

In [12]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    "district_heating",
    "energy_efficiency",
    "energy_grid",
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
    "energy_storage",    
    # "renewables_general",    
    "solar",
    "wind"
    # "decarbonisation_general",    
]

## Run the LLM checks

In [13]:
start_date = "2014-01-01"
end_date = "2025-03-31"

new_projects = (
    GTR.projects_enriched
    .query("(start >= @start_date and start <= @end_date)")
)
new_projects_text = GTR.get_projects_text().query("id in @new_projects.id.to_list()")

2025-04-03 18:01:23,385 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250309/projects.parquet
2025-04-03 18:01:32,739 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250309/projects.parquet
2025-04-03 18:01:33,851 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20250309/funds.parquet
2025-04-03 18:01:34,814 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20250309/funds.parquet


In [14]:
try:
    enrichment_df = (
        pd.read_csv(OUTPUT_DIR / "gtr_labelled_projects.csv")
        .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",")))
        .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",")))
    )

except FileNotFoundError:
    enrichment_df = kw.enrich_topic_labels(new_projects_text) # will take 5-6 minutes
    enrichment_df.to_csv(OUTPUT_DIR / "gtr_labelled_projects.csv", index=False)

In [15]:
from typing import List, Literal

def get_projects_in_nesta_categories(
    GTR,
    enrichment_df: pd.DataFrame,
    category_type: Literal["mission_labels", "topic_labels"],
    categories: List[str],
) -> pd.DataFrame:
    """Get all companies belonging to the provided categories"""
    matching_ids = (  # noqa
        enrichment_df
        .explode(category_type)
        .query(f"{category_type} in @categories")
        .id.to_list()
    )
    return GTR.projects_enriched.query("id in @matching_ids").drop_duplicates(subset="id")

def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_projects_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return get_projects_in_nesta_categories(GTR, enrichment_df, "topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = GTR.get_projects_text().query("id in @selected_df.id.to_list()")
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
    system_message = batch_check.generate_relevance_check_system_message(config)
    system_message += " If the text mentions the technology defined by the scope above only as a negative example (and instead focusses on a different technology), please mark it as 'no'."

    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]

    processor = batch_check.LLMProcessor(
        model_name="gpt-4o-mini",
        output_path=str(OUTPUT_DIR / f"gtr_llm_check_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.3)    

In [16]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        config = get_config_dict(config_name)
        selected_df = get_projects_from_config(config)
        logging.info(f"Checking relevance for {config_name} ({len(selected_df)} projects)")
        await check_relevance(selected_df, config_name, config)
        

In [17]:
await check_all_configs(CONFIG_NAMES)

2025-04-03 18:01:37,071 - root - INFO - Checking relevance for bioenergy (454 projects)
2025-04-03 18:01:37,880 - root - INFO - Using OpenAI
2025-04-03 18:01:37,935 - root - INFO - All data has already been processed.
2025-04-03 18:01:38,109 - root - INFO - Checking relevance for biomass_heating (11 projects)
2025-04-03 18:01:38,875 - root - INFO - Using OpenAI
2025-04-03 18:01:38,922 - root - INFO - All data has already been processed.
2025-04-03 18:01:39,089 - root - INFO - Checking relevance for built_environment (463 projects)
2025-04-03 18:01:39,874 - root - INFO - Using OpenAI
2025-04-03 18:01:39,921 - root - INFO - All data has already been processed.
2025-04-03 18:01:40,153 - root - INFO - Checking relevance for ccus (534 projects)
2025-04-03 18:01:41,037 - root - INFO - Using OpenAI
2025-04-03 18:01:41,091 - root - INFO - All data has already been processed.


CancelledError: 

## Spot check the results

In [ ]:
import pandas as pd
from discovery_utils.utils import google

sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"
tab_name = "ukri_check"

In [ ]:
n_samples = 10

gtr_cols = ["id", "title", "amount", "start", "url"]
final_cols = ["theme", "id", "title", "text", "amount", "start", "url", "model", "temperature", "is_relevant"]

llm_checks_df = []

for config_name in CONFIG_NAMES:
    # read a jsonl file
    llm_check_df = (
        pd.read_json(OUTPUT_DIR / f"gtr_llm_check_{config_name}.jsonl", lines=True)
        .merge(GTR.projects_enriched[gtr_cols], left_on="id", right_on="id", how="left")
        .merge(GTR.get_projects_text()[["id", "text"]], left_on="id", right_on="id", how="left")
        .assign(theme=config_name)
        .groupby("is_relevant")[final_cols]
        # sample n or less from each group
        .apply(lambda df: df.sample(n_samples) if len(df) > n_samples else df)
        .reset_index(drop=True)
    )[final_cols]
    llm_checks_df.append(llm_check_df)

llm_checks_df = pd.concat(llm_checks_df, ignore_index=True)


In [ ]:
google.upload_data_to_gsheet(sheet_id, {tab_name: llm_checks_df})
google.format_gsheet(sheet_id, tab_name, freeze_cols=2)

2025-03-19 06:30:45,161 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
2025-03-19 06:30:46,633 - root - INFO - Uploading DataFrame to sheet: ukri_check
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-03-19 06:31:03,258 - root - INFO - Upload completed successfully.
2025-03-19 06:31:03,850 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]


### Check the results

In [ ]:
checked_df = google.access_google_sheet(sheet_id, tab_name)